<a href="https://colab.research.google.com/github/chathuminakir-art/APIIT-BOT/blob/main/CB015656_APIIT_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q nltk ipywidgets
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
import re
import random
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import ipywidgets as widgets
from IPython.display import display

# ---------------------------
# Knowledge Base
# ---------------------------
pairs = [
    # --- Greetings ---
    [r"hi|hello|hey",
     ["Hello!", "Hi there!", "Hey!"]],

    [r"my name is (.*)",
     ["Nice to meet you, %1!"]],

    [r"how are you",
     ["I'm doing well, thank you!", "I'm great, how about you?"]],

    [r"i am (good|well|okay|ok)",
     ["That's good to hear!"]],

    [r"what do you do",
     ["I'm a chatbot — I can help answer questions or chat with you."]],

    [r"quit",
     ["Goodbye! Have a great day!"]],

    # --- Admissions Q&A (Task 02) ---
    [r"(.*)undergraduate program(.*)",
     ["APIIT offers a range of undergraduate programs in Computing, Business, and Law."]],
    [r"(.*)postgraduate(.*)",
     ["Yes, we do offer postgraduate degrees in selected fields."]],
    [r"(.*)entry requirement(.*)computing(.*)",
     ["The entry requirements for the Computing degree usually include A-Levels or equivalent with passes in Mathematics and English."]],
    [r"(.*)apply(.*)course(.*)",
     ["You can apply online via our website or visit the admissions office to submit your application."]],
    [r"(.*)next intake(.*)",
     ["Our next intake is scheduled for January and September each year."]],
    [r"(.*)part[- ]time(.*)",
     ["Yes, there are part-time options available for some programs."]],
    [r"(.*)tuition fee(.*)computer science(.*)",
     ["The tuition fee for the BSc (Hons) in Computer Science is available from our admissions office."]],
    [r"(.*)installment(.*)",
     ["Yes, APIIT allows tuition fee payments in installments."]],
    [r"(.*)scholarship(.*)",
     ["We do offer scholarships based on academic merit and other criteria."]],
    [r"(.*)registration fee(.*)",
     ["The registration fee for new students is around LKR 25,000 (please confirm with admissions)."]],
    [r"(.*)document(.*)submit(.*)",
     ["You need to submit your academic transcripts, birth certificate, NIC/Passport copy, and completed application form."]],
    [r"(.*)apply online(.*)",
     ["Yes, you can apply online through the APIIT website."]],
    [r"(.*)duration(.*)undergraduate(.*)",
     ["Most undergraduate degrees last 3 years (full-time)."]],
    [r"(.*)lecturer(.*)computing(.*)",
     ["Our School of Computing lecturers are highly qualified professionals. For specific profiles, please visit the website."]],
    [r"(.*)international student(.*)",
     ["Yes, international students are welcome to apply to APIIT."]],
    [r"(.*)english language(.*)requirement(.*)",
     ["International students need an IELTS score of at least 6.0 or equivalent."]],
    [r"(.*)application deadline(.*)",
     ["Applications usually close one month before the intake date."]],
    [r"(.*)foundation(.*)bridging(.*)",
     ["Yes, we offer a foundation program for students who do not meet the direct entry requirements."]],
    [r"(.*)career(.*)computing degree(.*)",
     ["Career opportunities include software engineering, data analysis, cybersecurity, and more."]],
    [r"(.*)campus visit(.*)",
     ["Yes, you can book a campus tour by contacting the admissions office."]],
]

# ---------------------------
# NLP utilities
# ---------------------------
ps = PorterStemmer()

def preprocess(user_input):
    """Tokenize + stem, return space-joined stems (for simple substring checks)."""
    tokens = word_tokenize(user_input.lower())
    stemmed = [ps.stem(token) for token in tokens]
    return " ".join(stemmed)

def extract_literal_words(pattern):
    """
    Take a regex-ish pattern string and extract plain words from it.
    E.g. "(.*)undergraduate program(.*)" -> ["undergraduate", "program"]
    """
    # remove escaped characters like \. or \?
    cleaned = re.sub(r'\\.', ' ', pattern)
    # replace common regex special characters with spaces
    cleaned = re.sub(r'[\(\)\^\$\.\*\?\+\|\[\]\{\}\\]', ' ', cleaned)
    # extract word tokens
    words = re.findall(r'\w+', cleaned)
    return words

class SmartChat:
    def __init__(self, pairs):
        self.pairs = pairs
        self.ps = PorterStemmer()

    def respond(self, user_input):
        # 1) try original regex-based matching (with capture replacement)
        for pattern, responses in self.pairs:
            try:
                match = re.search(pattern, user_input, re.IGNORECASE)
                if match:
                    response = random.choice(responses)
                    # Replace %1, %2, ... with captured groups
                    for i in range(len(match.groups())):
                        response = response.replace(f"%{i+1}", match.group(i+1))
                    return response
            except re.error:
                continue

        # 2) fallback: stemmed matching
        user_proc = preprocess(user_input)
        for pattern, responses in self.pairs:
            words = extract_literal_words(pattern)
            if not words:
                continue
            stemmed_words = [self.ps.stem(w.lower()) for w in words]
            if all(sw in user_proc for sw in stemmed_words):
                return random.choice(responses)

        # no match
        return None

# instantiate chatbot
chatbot = SmartChat(pairs)

# ---------------------------
# Colab widget UI (input box + output area)
# ---------------------------
input_box = widgets.Text(
    placeholder='Ask me about APIIT admissions...',
    description='You:',
    disabled=False,
    layout=widgets.Layout(width='95%')
)
output_box = widgets.Output(layout={'border': '1px solid #ddd'})

def handle_input(sender):
    user_message = input_box.value.strip()
    if user_message == "":
        return
    with output_box:
        print("You:", user_message)
        if user_message.lower() == "quit":
            print("Bot: Goodbye!\n")
            input_box.value = ''
            return
        response = chatbot.respond(user_message)
        if response:
            print("Bot:", response, "\n")
        else:
            print("Bot: Sorry, I don’t have an answer for that. Please contact admissions.\n")
    input_box.value = ''  # clear box

input_box.on_submit(handle_input)
display(input_box, output_box)
print("Chatbot ready! Type your question in the box (type 'quit' to exit).")

Text(value='', description='You:', layout=Layout(width='95%'), placeholder='Ask me about APIIT admissions...')

Output(layout=Layout(border='1px solid #ddd'))

Chatbot ready! Type your question in the box (type 'quit' to exit).


In [ ]:
!pip install pyngrok flask

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
!pip install flask-cors

In [ ]:
from pyngrok import ngrok, conf
ngrok.set_auth_token("32uiy3iTLVL1L0AuYiWypUWf9SQ_2CzpBW8uRbTyTTDGXK6Ji")

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

@app.route("/chat", methods=["POST"])
def chat():
    data = request.get_json()
    user_message = data.get("message", "")
    response = chatbot.respond(user_message)
    return jsonify({"response": response})

# --- start public server via ngrok ---
public_url = ngrok.connect(5000)
print("Public URL:", public_url)

app.run(port=5000)

Public URL: NgrokTunnel: "https://bb8b-35-201-155-132.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [26/Sep/2026 14:51:43] "OPTIONS / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [26/Sep/2026 14:52:01] "OPTIONS / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [26/Sep/2026 14:52:53] "OPTIONS /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Sep/2026 14:52:54] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Sep/2026 14:53:09] "OPTIONS /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Sep/2026 14:53:09] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Sep/2026 14:53:19] "OPTIONS /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Sep/2026 14:53:20] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Sep/2026 14:53:47] "OPTIONS /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Sep/2026 14:53:48] "POST /chat HTTP/1.1" 20